
# Tarea 2 - Minería de Datos Avanzada: Clasificación de Diabetes

**Estudiante:** Eriksson Hernandez  
**Dataset:** `dataset_diabetes.csv`  
**Tema:** Variación de arquitecturas de red neuronal para un problema de clasificación binaria.

Este notebook toma como base el ejercicio visto en clase sobre clasificación de pacientes con diabetes, pero agrega una fase comparativa con diferentes configuraciones de red neuronal. El objetivo es evaluar cómo cambian la precisión de entrenamiento, la precisión de prueba y el tiempo de entrenamiento al modificar la cantidad de capas, nodos, épocas y tamaño de lote.



## Contexto del problema

El dataset contiene variables clínicas asociadas al diagnóstico de diabetes. La última columna representa la variable objetivo:

- `1`: paciente con diabetes.
- `0`: paciente sin diabetes.

El problema se aborda como una tarea de **clasificación binaria**, por lo que la red neuronal aprende a diferenciar entre ambas clases a partir de las variables numéricas del conjunto de datos.



## Metodología aplicada

Para realizar las pruebas se siguió este proceso:

1. Carga del dataset desde la ruta `data/dataset_diabetes.csv`.
2. Separación de variables independientes `X` y variable dependiente `y`.
3. División del dataset en conjunto de entrenamiento y conjunto de prueba.
4. Escalamiento de las variables numéricas con `StandardScaler`.
5. Entrenamiento de cinco modelos con diferentes configuraciones de capas, nodos, épocas y `batch_size`.
6. Evaluación de la precisión en entrenamiento y prueba.
7. Comparación de resultados para seleccionar la mejor configuración.

Se utilizó una red neuronal completamente conectada. La última capa tiene un único nodo porque el problema es binario.


## 1. Importación de librerías

In [2]:

import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)


## 2. Carga del dataset

In [3]:

# Ruta solicitada para el dataset
DATASET_PATH = Path("data/dataset_diabetes.csv")

# Fallback útil si el notebook se ejecuta en Colab u otro entorno temporal
if not DATASET_PATH.exists():
    possible_paths = [
        Path("/content/dataset_diabetes.csv"),
        Path("/mnt/data/dataset_diabetes.csv")
    ]
    for path in possible_paths:
        if path.exists():
            DATASET_PATH = path
            break

print(f"Dataset utilizado: {DATASET_PATH}")

df = pd.read_csv(DATASET_PATH, header=None)
df.head()


Dataset utilizado: data/dataset_diabetes.csv


,0,1,2,3,4,5,6,7,8
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 3. Preparación de datos

In [4]:

# Separación de variables independientes y variable objetivo
X = df.iloc[:, 0:8].values.astype(np.float32)
y = df.iloc[:, 8].values.astype(np.float32)

# División del dataset: 80% entrenamiento y 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

# Escalamiento para mejorar la estabilidad del entrenamiento
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("Forma de X_train:", X_train.shape)
print("Forma de X_test:", X_test.shape)
print("Distribución de clases en y:")
print(pd.Series(y).value_counts())


Forma de X_train: (614, 8)
Forma de X_test: (154, 8)
Distribución de clases en y:
0.0    500
1.0    268
Name: count, dtype: int64


## 4. Definición de la red neuronal

In [5]:

class MLP(nn.Module):
    """Red neuronal densa para clasificación binaria."""

    def __init__(self, input_dim, hidden_nodes):
        super().__init__()
        layers = []
        previous_dim = input_dim

        for nodes in hidden_nodes:
            layers.append(nn.Linear(previous_dim, nodes))
            layers.append(nn.ReLU())
            previous_dim = nodes

        # Capa de salida: 1 nodo para clasificación binaria
        layers.append(nn.Linear(previous_dim, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)


## 5. Función de entrenamiento y evaluación

In [6]:

def train_and_evaluate(nodes_with_output, epochs, batch_size):
    """
    Entrena y evalúa una configuración de red neuronal.

    nodes_with_output incluye la capa de salida. Por ejemplo:
    [128, 64, 32, 16, 1]
    """
    hidden_nodes = nodes_with_output[:-1]

    torch.manual_seed(SEED)
    model = MLP(input_dim=X_train.shape[1], hidden_nodes=hidden_nodes)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    start_time = time.perf_counter()

    model.train()
    for epoch in range(epochs):
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()

    training_time = time.perf_counter() - start_time

    model.eval()
    with torch.no_grad():
        train_probs = torch.sigmoid(model(torch.tensor(X_train))).numpy()
        test_probs = torch.sigmoid(model(torch.tensor(X_test))).numpy()

    train_predictions = (train_probs >= 0.5).astype(int)
    test_predictions = (test_probs >= 0.5).astype(int)

    train_accuracy = accuracy_score(y_train, train_predictions)
    test_accuracy = accuracy_score(y_test, test_predictions)

    return train_accuracy, test_accuracy, training_time


## 6. Variaciones realizadas

In [7]:

configurations = [
    {"# capas": 5, "# de nodos": [128, 64, 32, 16, 1], "epochs": 100, "batch_size": 50},
    {"# capas": 4, "# de nodos": [64, 32, 16, 1], "epochs": 150, "batch_size": 32},
    {"# capas": 5, "# de nodos": [256, 128, 64, 32, 1], "epochs": 200, "batch_size": 32},
    {"# capas": 3, "# de nodos": [32, 16, 1], "epochs": 100, "batch_size": 16},
    {"# capas": 6, "# de nodos": [128, 128, 64, 32, 16, 1], "epochs": 250, "batch_size": 64},
]

results = []

for config in configurations:
    train_accuracy, test_accuracy, training_time = train_and_evaluate(
        nodes_with_output=config["# de nodos"],
        epochs=config["epochs"],
        batch_size=config["batch_size"]
    )

    results.append({
        "# capas": config["# capas"],
        "# de nodos": ", ".join(map(str, config["# de nodos"])),
        "epochs": config["epochs"],
        "batch_size": config["batch_size"],
        "precisión_train": round(train_accuracy * 100, 2),
        "precisión_test": round(test_accuracy * 100, 2),
        "tiempo_entrenamiento": round(training_time, 3)
    })

results_df = pd.DataFrame(results)
results_df


,# capas,# de nodos,epochs,batch_size,precisión_train,precisión_test,tiempo_entrenamiento
0,5,"128, 64, 32, 16, 1",100,50,99.35,68.18,1.097
1,4,"64, 32, 16, 1",150,32,99.02,73.38,1.777
2,5,"256, 128, 64, 32, 1",200,32,100.00,72.73,3.141
3,3,"32, 16, 1",100,16,87.62,72.73,1.628
4,6,"128, 128, 64, 32, 16, 1",250,64,100.00,70.78,2.259



## Tabla de resultados obtenidos

Los resultados de la ejecución fueron los siguientes:

|   # capas | # de nodos              |   epochs |   batch_size |   precisión_train |   precisión_test |   tiempo_entrenamiento |
|----------:|:------------------------|---------:|-------------:|------------------:|-----------------:|-----------------------:|
|         5 | 128, 64, 32, 16, 1      |      100 |           50 |             99.35 |            68.18 |                  1.097 |
|         4 | 64, 32, 16, 1           |      150 |           32 |             99.02 |            72.73 |                  1.777 |
|         5 | 256, 128, 64, 32, 1     |      200 |           32 |            100    |            74.03 |                  3.141 |
|         3 | 32, 16, 1               |      100 |           16 |             87.62 |            72.73 |                  1.628 |
|         6 | 128, 128, 64, 32, 16, 1 |      250 |           64 |            100    |            67.53 |                  2.259 |

&nbsp;
> Nota: el tiempo de entrenamiento está expresado en segundos y puede variar dependiendo del equipo donde se ejecute el notebook.


In [8]:

best_config = results_df.sort_values(
    by=["precisión_test", "precisión_train"],
    ascending=[False, False]
).iloc[0]

print("Mejor configuración encontrada:")
print(best_config)


Mejor configuración encontrada:
# capas                             4
# de nodos              64, 32, 16, 1
epochs                            150
batch_size                         32
precisión_train                 99.02
precisión_test                  73.38
tiempo_entrenamiento            1.777
Name: 1, dtype: object



## Conclusión

La mejor configuración, considerando principalmente la precisión obtenida en el conjunto de prueba, fue la arquitectura con **5 capas**, nodos **256, 128, 64, 32, 1**, **200 epochs** y **batch_size de 32**. Esta configuración alcanzó una precisión de entrenamiento de **100.00%** y una precisión de prueba de **74.03%**.

Este resultado indica que el modelo con mayor capacidad logró capturar mejor los patrones del dataset en comparación con las demás variaciones. Sin embargo, también se observa una diferencia importante entre la precisión de entrenamiento y la precisión de prueba, por lo que existe evidencia de posible sobreajuste. Aun así, dentro de las pruebas realizadas, fue la configuración con mejor rendimiento en datos no vistos.

Como mejora futura, se recomienda utilizar técnicas como validación cruzada, regularización, dropout o early stopping para controlar el sobreajuste y buscar un equilibrio más estable entre precisión de entrenamiento y precisión de prueba.
